# RQ II: User-response models


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reported_runs/analysis").is_dir())
sys.path.insert(0, str(repo)) if str(repo) not in sys.path else None
from reported_runs.analysis.presentation import load_results, NotebookReport, temporal_table, effect_table, rank_summary, unique_trajectories, label_conditions
from reported_runs.analysis.plotting import trajectory_grid, effect_grid, rank_grid, association_grid
from reported_runs.analysis.comparisons import summarise_iterations
%matplotlib inline

results = load_results()
report = NotebookReport(results, 'rq2')
scales = report.scales
trajectory = label_conditions(results["trajectory"])
sources = results["sources"]
summaries = results["summaries"]
effect_summaries = results["effect_summaries"]
pd.set_option("display.max_columns", 18)
pd.set_option("display.max_rows", 60)

## User-response models under common feedback conditions


In [ ]:
from reported_runs.analysis.sections_rq1_3 import diagnostic_tables, arm_rank_statistics
models = trajectory.loc[trajectory.study.eq('movielens_feedback_a300_e804') & trajectory.stage.str.startswith(('L0', 'L1', 'L2'))].copy()
model_effects = effect_summaries.loc[effect_summaries.study.eq('movielens_feedback_a300_e804') & effect_summaries.kind.eq('feedback')].copy()

## Click composition and selection relative to exposure


In [ ]:
for stage, selected in models.groupby('stage', sort=True):
    figure = trajectory_grid(selected, ['head_click_share', 'click_exposure_gap', 'weighted_click_exposure_gap'], title=f'User-response models: {stage.split("_")[0]}', scales=scales)
    report.figure(figure, selected, 'rq2_model_trajectories_' + stage.split('_')[0].lower(), "", recommended=False, why="")
diagnostic_tables(report, models, {'head_clicks': ['head_click_share'], 'exposure': ['head_exposure', 'weighted_head_exposure'], 'selection': ['click_exposure_gap', 'weighted_click_exposure_gap']}, prefix='rq2_models', context="")

In [ ]:
figure = effect_grid(model_effects, ['head_click_share', 'click_exposure_gap', 'weighted_click_exposure_gap'], title='Model-specific changes after adding feedback', scales=scales, value='value', facet='window')
report.figure(figure, model_effects, 'rq2_model_feedback_effects', "")
report.table(effect_table(model_effects, 'head_click_share'), 'rq2_model_feedback_head_clicks', "")

## Click behaviour, catalogue coverage, and concentration


In [ ]:
diagnostic_tables(report, models, {'click_behaviour': ['click_rate', 'no_click_rate', 'invalid_rate', 'valid_clicks'], 'coverage': ['exposure_coverage', 'click_coverage'], 'concentration': ['exposure_gini', 'click_gini']}, prefix='rq2_models', context="")

## Identical-order repeatability and order sensitivity


In [ ]:
probes = label_conditions(results['probes'].loc[results['probes'].study.eq('movielens_feedback_a300_e804') & results['probes'].stage.str.startswith(('L0', 'L1', 'L2'))])
llm_probes = probes.loc[probes.model.isin(['direct_llm', 'agentic_recommender'])]
computational_probes = probes.loc[~probes.model.isin(['direct_llm', 'agentic_recommender'])]
diagnostic_tables(report, llm_probes, {'repeatability': ['replica_pair_agreement', 'replica_both_no_click_share'], 'action_agreement': ['replica_action_agreement', 'replica_action_disagreement', 'shuffled_action_agreement', 'shuffled_action_disagreement'], 'common_subset_agreement': ['replica_agreement', 'shuffled_agreement', 'agreement_sensitivity'], 'eligibility': ['triple_valid_clicks', 'replica_both_clicks', 'shuffled_both_clicks'], 'valid_pairs': ['replica_valid_pairs', 'shuffled_valid_pairs'], 'excluded_pairs': ['replica_excluded_non_valid_pairs', 'shuffled_excluded_non_valid_pairs']}, prefix='rq2_llm', context="")
diagnostic_tables(report, computational_probes, {'order_sensitivity': ['shuffle_tvd', 'replica_tvd'], 'saved_probability_coverage': ['tvd_opportunities', 'triple_valid_clicks']}, prefix='rq2_computational', context="")

In [ ]:
for stage, selected in llm_probes.groupby('stage', sort=True):
    figure = trajectory_grid(selected, ['replica_agreement', 'shuffled_agreement', 'agreement_sensitivity'], title=f'LLM order diagnostics: {stage.split("_")[0]}', scales=scales)
    report.figure(figure, selected, 'rq2_llm_order_' + stage.split('_')[0].lower(), "")
for stage, selected in computational_probes.groupby('stage', sort=True):
    figure = trajectory_grid(selected, 'shuffle_tvd', title=f'Computational probability TVD: {stage.split("_")[0]}', scales=scales)
    report.figure(figure, selected, 'rq2_computational_tvd_' + stage.split('_')[0].lower(), "")

## Clicked positions


In [ ]:
ranks = results['ranks'].loc[results['ranks'].study.eq('movielens_feedback_a300_e804') & results['ranks'].stage.str.startswith(('L0', 'L1', 'L2'))]
for stage, selected in ranks.groupby('stage', sort=True):
    rank_counts = rank_summary(selected)
    figure = rank_grid(rank_counts, title=f'Clicked displayed positions: {stage.split("_")[0]}')
    report.figure(figure, rank_counts, 'rq2_clicked_ranks_' + stage.split('_')[0].lower(), "")
    report.table(rank_counts, 'rq2_rank_counts_' + stage.split('_')[0].lower(), "")
rank_statistics = arm_rank_statistics(ranks)
diagnostic_tables(report, rank_statistics, {'rank_statistics': ['mean_clicked_rank', 'first_position_click_share']}, prefix='rq2_models', context="", extra_ids=['arm'])

## Descriptive mean and consistency across available seeds


In [ ]:
from reported_runs.analysis.presentation import cross_seed_table
for metric in ['head_click_share', 'click_exposure_gap', 'weighted_click_exposure_gap']:
    table = cross_seed_table(results, 'rq2', [metric])
    report.table(table, f'rq2_seed_consistency_{metric}', "")